In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
msgs = [{"role":"user","content":"Hi"},{"role":"assistant","content":"Hello!"}]
text = tok.apply_chat_template(msgs, tokenize=False)
print(repr(text))

In [ ]:
enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
for t , off in zip(enc["input_ids"], enc["offset_mapping"]):
    print(t, off, repr(text[off[0]:off[1]]), type(off))

In [ ]:
from transformers import AutoTokenizer

msgs = [{"role": "user", "content": "Weather?"},
        {"role": "assistant", "reasoning_content": "User wants weather, call the tool.", "content": "Let me check."}]

for name in ["Qwen/Qwen3-4B", "unsloth/Llama-3.2-3B-Instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"====={name}\n{tok.apply_chat_template(msgs, tokenize=False)}")

### MODEL ADAPTERS TEST

In [ ]:
from transformers import AutoTokenizer
from sft.data.chat_tokenizer import get_adapter

In [ ]:
msgs = [{"role": "user", "content": "Weather?"},
        {"role": "assistant", "reasoning_content": "Call the tool.", "content": "Let me check.", "train": True}]
for name in ["Qwen/Qwen3-4B", "unsloth/Llama-3.2-3B-Instruct"]:
    a = get_adapter(AutoTokenizer.from_pretrained(name))
    out, tools = a.prepare(msgs, [])
    print(name, "|", a.name, a.reasoning, "|", out[1], "| tools:", tools, "| kwargs:", a.template_kwargs(out))
print("input unchanged:", msgs[1])

In [ ]:
from transformers import AutoTokenizer
from sft.data.chat_tokenizer import ChatTokenizer, _rstrip_end
ct = ChatTokenizer(AutoTokenizer.from_pretrained("Qwen/Qwen3-4B"))
tools = [{"type":"function","function":{"name":"get_weather","description":"Get weather.","parameters":{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}}}]
msgs = [{"role":"user","content":"Weather in Paris?"},
 {"role":"assistant","reasoning_content":"Call the tool.","content":"","tool_calls":[{"id":"c1","type":"function","function":{"name":"get_weather","arguments":{"city":"Paris"}}}]},
 {"role":"tool","tool_call_id":"c1","name":"get_weather","content":"18C, sunny"},
 {"role":"assistant","reasoning_content":"Got it.","content":"It's 18C and sunny in Paris."}]


In [ ]:
m, t = ct.adapter.prepare(msgs, tools)
kw = ct.adapter.template_kwargs(m)

In [ ]:
for i in (1, 3):
    p, u = ct._turn_span(m, t, kw, i)
    e = _rstrip_end(u, len(p), len(u))
    print(i, "prefix ok:", u.startswith(p), "| span:", repr(u[len(p):e]), "| stripped:", repr(u[e:]))

### CHAT_TOKENIZE.PY full test

In [ ]:
import copy
from transformers import AutoTokenizer
from sft.data.chat_tokenizer import ChatTokenizer, TokenizerConfig
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
tools = [{"type":"function","function":{"name":"get_weather","description":"Get weather.","parameters":{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}}}]
base = {"id": "r1", "metadata": {"source": "test", "task_type": "tool_calling"}, "tools": tools, "messages": [
 {"role":"user","content":"Weather in Paris?"},
 {"role":"assistant","reasoning_content":"Call the tool.","content":"","tool_calls":[{"id":"c1","type":"function","function":{"name":"get_weather","arguments":{"city":"Paris"}}}]},
 {"role":"tool","tool_call_id":"c1","name":"get_weather","content":"18C, sunny"},
 {"role":"assistant","reasoning_content":"Got it.","content":"It's 18C and sunny in Paris."}]}

def show(name, rec, **cfg):
    r = ChatTokenizer(tok, TokenizerConfig(**cfg))(rec)
    print(f"{name:22} mode={r.mode!s:9} samples={len(r.samples)} fallback={r.fallback_reason!r} "
          f"rejections={[(x.reason.value, x.turn_idx) for x in r.rejections]}")

no_r = copy.deepcopy(base); [m.pop("reasoning_content", None) for m in no_r["messages"]]
frozen = copy.deepcopy(base); [m.__setitem__("train", False) for m in frozen["messages"]]
half = copy.deepcopy(base); half["messages"][1]["train"] = False

show("reasoning (auto)", base)                       # single, 1 sample
show("no_reasoning (auto)", no_r)                    # per_turn, 2 samples, fallback set
show("no_reasoning (single)", no_r, mode="single")   # rejected prefix_mismatch
show("reasoning (per_turn)", base, mode="per_turn")  # per_turn, 2 samples, fallback None
show("overlong 200 (auto)", base, max_seq_len=200)   # per_turn, 1 sample + 1 overlong rejection
show("all frozen", frozen)                           # rejected no_train_turns
show("turn 1 frozen", half)                          # single, 1 sample
show("no metadata", {"id": "x", "messages": base["messages"]})   # works
show("missing messages", {"id": "bad1"})                         # rejected, must NOT crash
show("missing role", {"id": "bad2", "messages": [{"content": "hi"}]})  # rejected, must NOT crash
before = copy.deepcopy(base); ChatTokenizer(tok)(base); print("input unchanged:", before == base)
try:
    ChatTokenizer(tok, TokenizerConfig(raise_errors=True))({"id": "bad1"}); print("raise_errors: did NOT raise ❌")
except KeyError as e:
    print("raise_errors: raised KeyError ✅", e)

In [ ]:
%load_ext autoreload
%autoreload 2


In [2]:
from tests.conftest import make_tiny_model
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
model = make_tiny_model(len(tok))

In [3]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151669, 64)
    (layers): ModuleList(
      (0-1): 2 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=64, out_features=64, bias=False)
          (k_proj): Linear(in_features=64, out_features=32, bias=False)
          (v_proj): Linear(in_features=64, out_features=32, bias=False)
          (o_proj): Linear(in_features=64, out_features=64, bias=False)
          (q_norm): Qwen3RMSNorm((16,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((16,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=64, out_features=128, bias=False)
          (up_proj): Linear(in_features=64, out_features=128, bias=False)
          (down_proj): Linear(in_features=128, out_features=64, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((64,), eps=1e-06)
        (post_attention_layernorm): Qwen3RMSNorm((64,), eps=1e

In [4]:
model.model.layers

ModuleList(
  (0-1): 2 x Qwen3DecoderLayer(
    (self_attn): Qwen3Attention(
      (q_proj): Linear(in_features=64, out_features=64, bias=False)
      (k_proj): Linear(in_features=64, out_features=32, bias=False)
      (v_proj): Linear(in_features=64, out_features=32, bias=False)
      (o_proj): Linear(in_features=64, out_features=64, bias=False)
      (q_norm): Qwen3RMSNorm((16,), eps=1e-06)
      (k_norm): Qwen3RMSNorm((16,), eps=1e-06)
    )
    (mlp): Qwen3MLP(
      (gate_proj): Linear(in_features=64, out_features=128, bias=False)
      (up_proj): Linear(in_features=64, out_features=128, bias=False)
      (down_proj): Linear(in_features=128, out_features=64, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): Qwen3RMSNorm((64,), eps=1e-06)
    (post_attention_layernorm): Qwen3RMSNorm((64,), eps=1e-06)
  )
)